# CREST Phase 1: Reproduce Tables and Figures

This notebook regenerates the Phase 1 benchmark tables and figures included in the CREST Phase 1 technical package.

**Scope.** This notebook reproduces the document-facing tables/figures from the Phase 1 asset CSVs. It does **not** rerun the full raw SPARC fitting pipeline. A later notebook should add raw SPARC ingestion, unit conversion, grouped galaxy-held-out folds, MOND/RAR comparators, and residual-layer fitting from first principles.

Expected repository layout:

```text
.
├── CREST_Phase1_Reproducibility_Assets.zip
├── assets/
├── data/
├── docs/
├── notebooks/
└── src/
```

The notebook can find the asset CSVs either inside the ZIP file or in an extracted folder.

In [ ]:
from pathlib import Path
import zipfile
import shutil

import pandas as pd
import matplotlib.pyplot as plt

# Resolve likely project root. If run from /notebooks, use parent; otherwise use current directory.
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
print(f"Current working directory: {cwd}")
print(f"Project root assumed as: {project_root}")

output_dir = project_root / "outputs"
fig_dir = output_dir / "figures"
table_dir = output_dir / "tables"
fig_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)
print(f"Outputs will be written to: {output_dir}")

## 1. Locate or extract the Phase 1 asset CSVs

The current release stores the reproducibility materials as `CREST_Phase1_Reproducibility_Assets.zip`. This cell extracts the ZIP if available and then searches for the three required CSV files.

In [ ]:
asset_zip_candidates = [
    project_root / "CREST_Phase1_Reproducibility_Assets.zip",
    project_root / "assets" / "CREST_Phase1_Reproducibility_Assets.zip",
    cwd / "CREST_Phase1_Reproducibility_Assets.zip",
]

extract_dir = project_root / "extracted_phase1_assets"
for zpath in asset_zip_candidates:
    if zpath.exists():
        print(f"Found asset ZIP: {zpath}")
        extract_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall(extract_dir)
        print(f"Extracted assets to: {extract_dir}")
        break
else:
    print("No asset ZIP found. Searching for already-extracted CSV files.")

required = {
    "rmse": "phase1_rmse_table.csv",
    "halo": "phase1_halo_reference_table.csv",
    "overlap": "phase1_overlap_cases.csv",
}

search_roots = [project_root, project_root / "assets", extract_dir, cwd]
found = {}
for key, filename in required.items():
    matches = []
    for root in search_roots:
        if root.exists():
            matches.extend(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find required file: {filename}")
    found[key] = matches[0]
    print(f"{key}: {found[key]}")

## 2. Load benchmark tables

In [ ]:
rmse = pd.read_csv(found["rmse"])
halo = pd.read_csv(found["halo"])
overlap = pd.read_csv(found["overlap"])

print("RMSE table")
display(rmse)

print("Halo reference table")
display(halo)

print("Overlap cases")
display(overlap)

## 3. Regenerate document-facing CSV outputs

In [ ]:
rmse.to_csv(table_dir / "phase1_rmse_table_regenerated.csv", index=False)
halo.to_csv(table_dir / "phase1_halo_reference_table_regenerated.csv", index=False)
overlap.to_csv(table_dir / "phase1_overlap_cases_regenerated.csv", index=False)
print("Wrote regenerated tables:")
for path in sorted(table_dir.glob("*.csv")):
    print("-", path)

## 4. Figure 1 — Full held-out RMSE comparison

Lower RMSE is better. The fixed CREST backbone alone does not beat the fixed MOND/RAR comparators, while the shared CREST Huber residual layer gives the lowest full-sample RMSE in this benchmark table.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.bar(rmse["Model"], rmse["Full RMSE"])
ax.set_ylabel("Grouped held-out RMSE in log10 g")
ax.set_title("Phase 1 Full RMSE Comparison")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig_path = fig_dir / "figure_1_full_rmse_regenerated.png"
fig.savefig(fig_path, dpi=200)
plt.show()
print(f"Saved: {fig_path}")

## 5. Figure 2 — Q-stratified RMSE comparison

This plot shows that the residual-layer improvement is not uniform. The largest improvement appears in the more complex Q = 2 and Q = 3 subsets, while Q = 1 remains essentially tied.

In [ ]:
q_cols = ["Q=1", "Q=2", "Q=3"]
fig, ax = plt.subplots(figsize=(8, 4.8))
for _, row in rmse.iterrows():
    ax.plot(q_cols, [row[c] for c in q_cols], marker="o", label=row["Model"])
ax.set_ylabel("Grouped held-out RMSE in log10 g")
ax.set_title("Phase 1 Q-Stratified RMSE")
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()
fig_path = fig_dir / "figure_2_q_stratified_rmse_regenerated.png"
fig.savefig(fig_path, dpi=200)
plt.show()
print(f"Saved: {fig_path}")

## 6. Figure 3 — Tuned halo reference comparison

The tuned pISO and NFW halo references are in-sample galaxy-by-galaxy reference fits. They are not the same kind of grouped predictive baseline, but they show the remaining empirical gap between a fixed response law and flexible per-galaxy halo templates.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.bar(halo["Model"], halo["Median RMS km/s"])
ax.set_ylabel("Median per-galaxy RMS (km/s)")
ax.set_title("Phase 1 Tuned Halo Reference Fits")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig_path = fig_dir / "figure_3_halo_reference_regenerated.png"
fig.savefig(fig_path, dpi=200)
plt.show()
print(f"Saved: {fig_path}")

## 7. Figure 4 — Phase 1 workflow schematic

In [ ]:
steps = [
    "SPARC/RAR inputs",
    "Fixed baryonic model",
    "CREST backbone",
    "MOND/RAR comparators",
    "Shared residual layer",
    "Grouped held-out RMSE",
    "Q-stratified interpretation",
]

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.axis("off")
for i, step in enumerate(steps):
    x = i / (len(steps) - 1)
    ax.text(x, 0.55, step, ha="center", va="center", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.35", fill=False))
    if i < len(steps) - 1:
        ax.annotate("", xy=((i+0.82)/(len(steps)-1), 0.55), xytext=((i+0.18)/(len(steps)-1), 0.55),
                    arrowprops=dict(arrowstyle="->"))
ax.set_title("CREST Phase 1 Validation Workflow")
fig.tight_layout()
fig_path = fig_dir / "figure_4_phase1_workflow_regenerated.png"
fig.savefig(fig_path, dpi=200)
plt.show()
print(f"Saved: {fig_path}")

## 8. Numerical interpretation checks

In [ ]:
full = dict(zip(rmse["Model"], rmse["Full RMSE"]))
crest_residual = full["CREST Huber residual layer"]
mond = full["Fixed MOND-style branch"]
rar = full["Fixed RAR-form branch"]
backbone = full["CREST backbone"]

improve_vs_mond = (mond - crest_residual) / mond * 100
improve_vs_rar = (rar - crest_residual) / rar * 100
improve_vs_backbone = (backbone - crest_residual) / backbone * 100

summary = pd.DataFrame({
    "Comparison": [
        "CREST residual layer vs CREST backbone",
        "CREST residual layer vs fixed MOND-style branch",
        "CREST residual layer vs fixed RAR-form branch",
    ],
    "Percent RMSE reduction": [improve_vs_backbone, improve_vs_mond, improve_vs_rar],
})
summary.to_csv(table_dir / "phase1_interpretation_summary.csv", index=False)
display(summary)

print("Interpretation:")
print(f"- Residual layer improves full RMSE vs backbone by {improve_vs_backbone:.2f}%.")
print(f"- Residual layer improves full RMSE vs fixed MOND-style branch by {improve_vs_mond:.2f}%.")
print(f"- Residual layer improves full RMSE vs fixed RAR-form branch by {improve_vs_rar:.2f}%.")

## 9. Output inventory

In [ ]:
print("Regenerated figures:")
for path in sorted(fig_dir.glob("*.png")):
    print("-", path.relative_to(project_root))

print("\nRegenerated tables:")
for path in sorted(table_dir.glob("*.csv")):
    print("-", path.relative_to(project_root))

## 10. Next notebook to add

The next notebook should move beyond document-facing reproduction and rerun the raw validation from public SPARC/RAR inputs:

1. Load SPARC rotation-curve and mass-model files.
2. Convert velocities/radii into acceleration observables.
3. Construct the fixed baryonic acceleration `gbar`.
4. Compute the CREST backbone prediction.
5. Fit MOND/RAR comparator parameters on training galaxies only.
6. Add the shared residual-layer protocol.
7. Evaluate grouped galaxy-held-out folds.
8. Reproduce the Phase 1 benchmark table from raw inputs.